#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "CID":"customer_id",
    "CNTRY":"country"
}

In [0]:
df = spark.table("data_lakehouse_project.bronze.bronze_log_a101")

#Data transform

In [0]:
#trim string value
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
#Australia, US, Canada, Germany,France, UK, "", null
df = df.withColumn(
    "CNTRY",
    F.when((F.upper(col("CNTRY")) == "US") | (F.upper(col("CNTRY")) == "USA"),"United States")
    .when(F.upper(col("CNTRY")) == "DE", "Germany")
    .when((F.upper(col("CNTRY")) == "") | (F.upper(col("CNTRY")).isNull()), "Unknown")
    .otherwise(col("CNTRY"))
)

In [0]:
#Remove - 
df = df.withColumn(
    "CID",
    F.expr("Replace(CID,'-','')")
)

#Write to silver layer

In [0]:
# Renaming columns header
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("data_lakehouse_project.silver.silver_log_a101")
)

In [0]:
%sql
Select
* 
from data_lakehouse_project.silver.silver_log_a101